# A_v (vertex/star) symmetry breaking vs $h_z$ — trained NQS at L=4

**Question.** At $h_z=0$ the toric-code ground state obeys the exact vertex symmetry
$A_v\,|\psi\rangle=|\psi\rangle$, i.e. in the $\sigma^z$ basis
$$\psi(A_v\!\cdot\!\sigma)=\psi(\sigma),$$
where $A_v$ flips the spins on the qubits touching vertex $v$. At $h_z\neq 0$ the field
does **not** commute with $A_v$ (since $\{\sigma^x,\sigma^z\}=0$), so the symmetry is
*physically* broken. This notebook measures **how strongly** the trained networks break it,
as a function of $h_z$.

**Method.** For each trained network draw configurations $\sigma$ and, for every vertex star,
form the log-amplitude difference
$$\delta_v(\sigma)=\log\psi(A_v\sigma)-\log\psi(\sigma),\qquad r_v=e^{\delta_v}=\psi(A_v\sigma)/\psi(\sigma).$$
This is **normalization-free** (a difference of log-amplitudes on the same net). Two config sets:
* **Born** $\sigma\sim|\psi|^2$ from the model's own MCMC sampler — physical primary.
  For real $\psi$, $\;\langle A_v\rangle = \mathbb{E}_{\rm born}[r_v]$ (=1 at $h_z=0$, dropping with $h_z$).
* **Uniform** random $\pm1$ — a tail stress-test (probes $\psi$ as a bare function).

**Data.** The 7 trained L=4 CNN-baseline networks ($h_z\in\{0,0.05,\dots,0.30\}$) are cloned
from the repo (`results/nqs/*.mpack`). **Each network is evaluated in its own cell** (§7) —
Colab is flaky when several `.mpack` loads share one cell.

## Before you start
1. **Runtime → Change runtime type → GPU** (any GPU is fine; L=4 is tiny).
2. Have a **GitHub token** with read access to the private repo ready.
3. Run top-to-bottom. If the install prompts a **runtime restart**, do it, then resume from
   *Mount Google Drive*.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install the pinned stack
`requirements.txt` pins a **CPU** `jaxlib` (for the NERSC conda env); on Colab we install the
**CUDA 12** build of the same `jax==0.5.2` instead, plus `netket==3.16.1.post1`, `flax==0.10.4`.

⚠️ If pip swaps a `jaxlib` already loaded this session, Colab asks you to **restart the runtime** —
do it, then resume from the *Mount Google Drive* cell (do NOT re-run the install).

In [ ]:
%pip install -q "jax[cuda12]==0.5.2" "jaxlib==0.5.1" netket==3.16.1.post1 flax==0.10.4
print("\nInstall finished. If a 'restart runtime' notice appeared, do Runtime > Restart session,")
print("then continue from the 'Mount Google Drive' cell (do NOT re-run this install cell).")

In [ ]:
import jax, jaxlib, netket, flax
print("jax", jax.__version__, "| jaxlib", jaxlib.__version__,
      "| netket", netket.__version__, "| flax", flax.__version__)
print("devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), \
    "No GPU visible to JAX - set Runtime>GPU and/or re-check the jax[cuda12] install."
print("OK: GPU visible.")

## 3. Mount Google Drive  *(resume point after any runtime restart)*
The cloned repo lives under `MyDrive/2D-TC-colab/` so it survives disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone (or update) the repo into Drive
Private repo → paste a GitHub token when prompted (used only for the clone, then stripped from the
stored remote). **`v3-complex-readout`** has the trained L=4 baselines (`results/nqs/*.mpack`), the
transformer-arm runs, and all the `model/` code (so the `SUFFIX` knob in §5 works for every arm).

In [ ]:
import os, sys, glob, getpass, subprocess

DRIVE_ROOT = "/content/drive/MyDrive/2D-TC-colab"   # persists across sessions
BRANCH     = "v3-complex-readout"                    # baselines + transformer arms + full model code
REPO_DIR   = os.path.join(DRIVE_ROOT, "2D-TC")
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    tok = getpass.getpass("GitHub token (repo read access): ")
    url = f"https://{tok}@github.com/SanzharBissenali/2D-TC.git"
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin",
                    "https://github.com/SanzharBissenali/2D-TC.git"], check=True)
    del tok, url
else:
    print("repo already present; trying a fast-forward pull (harmless if it fails)")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
assert os.path.exists("model/networks.py"), "model code missing - wrong branch?"
mpacks = sorted(glob.glob("results/nqs/G-equiv_1_L4_hx0.00_hz*.mpack"))
print("repo:", REPO_DIR)
print("L4 mpack files found:", len(mpacks))
assert mpacks, "no L4 trained networks in results/nqs/ - wrong branch?"
print("OK: trained networks present.")

## 5. Setup — helpers (netket-independent parts mirror `scripts/symmetry_check.py`)
Imports the repo's `model/` + `utils/` and defines the evaluation used by the per-network cells.
The metric aggregation `symmetry_breaking_metric` is the one physics knob — edit it to taste.

In [ ]:
import json, os, glob, re
import numpy as np
import jax.numpy as jnp
import netket as nk
from model.geometry import ToricCodeGeometry
from model.networks import KernelManager, create_model
from utils.io import load_model

# ---- knobs ----
LX             = 4
SUFFIX         = ""        # "" = plain CNN baseline; "tf"/"tfg"/... for a transformer arm
N_SAMPLES      = 8192      # configs per network (Born & uniform)
N_DISCARD      = 64        # MCMC burn-in per chain for Born sampling
SEED           = 0         # rng seed for the uniform configs
N_BINS         = 81
HIST_HALFWIDTH = 4.0
OUT_DIR        = os.path.join(os.getcwd(), "results", "symmetry")
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS = {}   # hz -> result dict, filled by the per-network cells below


def symmetry_breaking_metric(delta, ratio):
    """Aggregate per-(config, vertex) breaking into scalars.

    delta, ratio: real arrays (n_configs, n_vertices); ratio = exp(delta).
    EDIT ME: this defines "how strongly broken". Options (all -> 0 / Av->1 at hz=0):
      mean_abs_delta = mean|delta|;  rms_delta = sqrt(mean delta^2);
      mean_abs_ratio_dev = mean|r-1|;  Av_mean = mean r (= <A_v> for Born sigma).
    """
    n = delta.size
    return {
        "mean_abs_delta": float(np.mean(np.abs(delta))),
        "rms_delta": float(np.sqrt(np.mean(delta ** 2))),
        "max_abs_delta": float(np.max(np.abs(delta))) if n else 0.0,
        "mean_abs_ratio_dev": float(np.mean(np.abs(ratio - 1.0))),
        "Av_mean": float(np.mean(ratio)),
        "Av_sem": float(np.std(ratio) / np.sqrt(max(n, 1))),
    }


def config_from_sim_params(js):
    sp = js["sim_params"]
    def one(key, default=None):
        v = sp.get(key, default)
        return v[0] if isinstance(v, list) and len(v) else v
    cfg = {
        "architecture": one("architecture_type", "Combo"),
        "channels_noninv": list(sp["n_chann_noninv"]),
        "channels_inv": list(sp["n_chann_inv"]),
        "kernel_size": one("kernel_size_noninv"),
        "kernel_size_inv": one("kernel_size_inv"),
        "Lx": one("Lx"), "Ly": one("Lx"), "bc": one("BC", "OBC"),
        "dtype": one("param_dtype", "float64"), "rescale": one("rescale", 1.0),
        "symmetric_block": one("symmetric_block", "cnn"), "seed": one("seed", 0),
        "n_chains": one("n_chains", 16), "n_samples_saved": one("n_samples"),
        "n_sweeps": one("n_sweeps"), "n_discard": one("n_discard_per_chain=", 8),
        "chunk_size": one("chunk_size=", 2048),
    }
    for k in ("tf_layers", "tf1_layers", "tf_gamma_init", "tf_dmodel", "tf_heads",
              "tf_ffn_mult", "tf_activation", "tf_readout_K", "tf_complex_output"):
        if k in sp:
            cfg[k] = one(k)
    return cfg


def load_run(base, cfg):
    """Rebuild geometry+model and restore params. Returns (geometry, model, vstate)."""
    geometry = ToricCodeGeometry(cfg["Lx"], cfg["Ly"], cfg["bc"])
    hi = nk.hilbert.Spin(s=1 / 2, N=geometry.N)
    km = KernelManager(Lx=cfg["Lx"], Ly=cfg["Ly"], bc=cfg["bc"],
                       kernel_size=cfg["kernel_size"], kernel_size_inv=cfg["kernel_size_inv"],
                       arr_coord=geometry.arr_coord, dg_p=geometry.dg_p, N=geometry.N)
    model = create_model(cfg, geometry.plaq_all, km)
    n_chains = int(cfg["n_chains"])
    n_sweeps = int(cfg["n_sweeps"]) if cfg["n_sweeps"] is not None else geometry.N // 2
    # faithful (vertex-flip) sampler if available; else LocalRule (also samples |psi|^2)
    try:
        from simulation.custom_sampler import create_custom_sampler
        if len(geometry.vertex_bulk_hetero) == 0:
            raise RuntimeError("no bulk stars")
        sa = create_custom_sampler(geometry, hi, {"n_chains": n_chains, "n_sweeps": n_sweeps})
    except Exception:
        sa = nk.sampler.MetropolisSampler(hi, rule=nk.sampler.rules.LocalRule(),
                                          n_chains=n_chains, n_sweeps=n_sweeps, dtype=jnp.int8)
    n_samp_saved = int(cfg["n_samples_saved"]) if cfg.get("n_samples_saved") else n_chains * 8
    vs = load_model(base, sa, model, n_samples=n_samp_saved,
                    n_discard_per_chain=int(cfg["n_discard"]), chunk_size=int(cfg["chunk_size"]))
    return geometry, model, vs


def deltas_for_configs(model, params, X, stars):
    """delta[i,k] = log psi(flip star_k of config i) - log psi(config i), real."""
    X = np.asarray(X, dtype=np.float64)
    base = np.asarray(model.apply({"params": params}, X)).real
    out = np.empty((X.shape[0], len(stars)), dtype=np.float64)
    for k, star in enumerate(stars):
        Xf = X.copy()
        Xf[:, np.asarray(star)] *= -1.0
        out[:, k] = np.asarray(model.apply({"params": params}, Xf)).real - base
    return out


def summarize(delta, hist_range, n_bins):
    if delta.size == 0:
        return {}, {"edges": [], "counts": []}
    ratio = np.exp(delta)
    counts, edges = np.histogram(delta.ravel(), bins=n_bins, range=hist_range)
    return symmetry_breaking_metric(delta, ratio), {"edges": edges.tolist(), "counts": counts.tolist()}


def base_for_hz(hz):
    """Path (no extension) of the baseline run for this hz (SUFFIX-aware)."""
    end = ("_" + SUFFIX) if SUFFIX else ""
    for jp in glob.glob(f"results/nqs/G-equiv_1_L{LX}_hx0.00_hz*{end}.json"):
        name = os.path.basename(jp)[:-5]
        m = re.search(rf"_hz([0-9]+\.[0-9]+){re.escape(end)}$", name)
        if m and abs(float(m.group(1)) - hz) < 1e-6 and os.path.exists(jp[:-5] + ".mpack"):
            return jp[:-5]
    raise FileNotFoundError(f"no baseline run for hz={hz} (suffix={SUFFIX!r})")


def evaluate_run(hz):
    """Full symmetry evaluation of ONE trained network; stores + writes + returns result."""
    base = base_for_hz(hz)
    js = json.load(open(base + ".json"))
    cfg = config_from_sim_params(js)
    dt = cfg["dtype"]
    if dt != "float64":
        print(f"SKIP hz={hz}: dtype={dt!r} (complex path out of scope)")
        return None
    geometry, model, vs = load_run(base, cfg)
    params, N = vs.parameters, geometry.N
    bulk = [list(map(int, s)) for s in geometry.vertex_bulk_hetero]
    edge = [list(map(int, s)) for s in geometry.vertex_edge_hetero]
    # Born configs (sigma ~ |psi|^2) from the model's own sampler
    vs.n_discard_per_chain = int(N_DISCARD)
    vs.n_samples = int(N_SAMPLES)
    vs.reset()
    X_born = np.asarray(vs.samples).reshape(-1, N)[:N_SAMPLES]
    # Uniform +-1 configs (same count, fixed seed)
    X_unif = np.random.default_rng(SEED).choice([-1.0, 1.0], size=(N_SAMPLES, N))
    hr = (-HIST_HALFWIDTH, HIST_HALFWIDTH)
    e_last = float(np.real(complex(js["energy"][-1]))) if js.get("energy") else None
    res = {"L": int(cfg["Lx"]), "hz": float(hz), "N": N, "dtype": dt,
           "n_samples": int(N_SAMPLES), "n_vertices_bulk": len(bulk), "n_vertices_edge": len(edge),
           "file": os.path.basename(base), "arm": SUFFIX or "cnn-baseline",
           "energy_last": e_last, "n_iters": len(js.get("energy", []))}
    for tag, X in (("born", X_born), ("unif", X_unif)):
        for vset, stars in (("bulk", bulk), ("edge", edge)):
            if not stars:
                continue
            m, h = summarize(deltas_for_configs(model, params, X, stars), hr, N_BINS)
            res[f"{tag}_{vset}"] = {"metric": m, "hist": h}
    suff = f"_{SUFFIX}" if SUFFIX else ""
    out = os.path.join(OUT_DIR, f"sym_L{res['L']}_hz{hz:.3f}{suff}.json")
    json.dump(res, open(out, "w"), indent=2)
    RESULTS[round(float(hz), 5)] = res
    bb = res.get("born_bulk", {}).get("metric", {})
    print(f"hz={hz:.2f}  <A_v>={bb.get('Av_mean', float('nan')):.4f}  "
          f"mean|delta|={bb.get('mean_abs_delta', float('nan')):.3e}  ->  {os.path.basename(out)}")
    return res

print("helpers ready. N_SAMPLES =", N_SAMPLES, "| output ->", OUT_DIR)

## 6. (optional) list the networks that will be evaluated

In [ ]:
for hz in [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    try:
        print(f"hz={hz:.2f}  ->  {os.path.basename(base_for_hz(hz))}")
    except FileNotFoundError as e:
        print(f"hz={hz:.2f}  ->  MISSING ({e})")

## 7. Evaluate each network — **one cell per network**
Run these one at a time. Each cell independently loads a single `.mpack`, samples, and writes its
`results/symmetry/sym_L4_hz*.json`. If one hiccups, just re-run that cell — the others are unaffected.
The first cell also triggers JIT compilation (a few seconds); later cells reuse it and are fast.

In [ ]:
# --- Network hz=0.00 (independent; re-run alone if needed) ---
evaluate_run(0.00)

In [ ]:
# --- Network hz=0.05 (independent; re-run alone if needed) ---
evaluate_run(0.05)

In [ ]:
# --- Network hz=0.10 (independent; re-run alone if needed) ---
evaluate_run(0.10)

In [ ]:
# --- Network hz=0.15 (independent; re-run alone if needed) ---
evaluate_run(0.15)

In [ ]:
# --- Network hz=0.20 (independent; re-run alone if needed) ---
evaluate_run(0.20)

In [ ]:
# --- Network hz=0.25 (independent; re-run alone if needed) ---
evaluate_run(0.25)

In [ ]:
# --- Network hz=0.30 (independent; re-run alone if needed) ---
evaluate_run(0.30)

## 8. Aggregate + plot
Loads every `results/symmetry/sym_L4_hz*.json` (so it works even after a restart) and draws:
**A** breaking metric vs $h_z$ (Born vs uniform), **B** $\langle A_v\rangle$ vs $h_z$,
**C** the $\delta_v$ distributions. Figures also saved to `figures/`.

In [ ]:
import json, glob, os
import numpy as np, matplotlib.pyplot as plt

SYM_DIR = os.path.join(os.getcwd(), "results", "symmetry")
FIG_DIR = os.path.join(os.getcwd(), "figures")
os.makedirs(FIG_DIR, exist_ok=True)
ARM = SUFFIX or "cnn-baseline"
L = LX

runs = []
for f in sorted(glob.glob(os.path.join(SYM_DIR, "sym_L*_hz*.json"))):
    d = json.load(open(f))
    if d.get("arm", "cnn-baseline") == ARM and d["L"] == L:
        runs.append(d)
runs.sort(key=lambda d: d["hz"])
print(f"loaded {len(runs)} runs:", [round(r["hz"], 3) for r in runs])

def series(runs, tag, vset, key):
    xs, ys = [], []
    for r in runs:
        blk = r.get(f"{tag}_{vset}")
        if blk and key in blk["metric"]:
            xs.append(r["hz"]); ys.append(blk["metric"][key])
    return np.array(xs), np.array(ys)

# Fig A: breaking metric vs hz (Born vs uniform)
KEY = "mean_abs_delta"   # swap for "rms_delta" / "mean_abs_ratio_dev"
fig, ax = plt.subplots(figsize=(6, 4))
for tag, style in (("born", "-o"), ("unif", "--s")):
    x, y = series(runs, tag, "bulk", KEY)
    if len(x):
        ax.plot(x, y, style, label=f"{tag} (bulk)")
ax.set_xlabel(r"$h_z$")
ax.set_ylabel(r"$\langle|\delta_v|\rangle$")
ax.set_title(f"$A_v$ symmetry breaking vs $h_z$  (L={L}, {ARM})")
ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"08_breaking_vs_hz_L{L}.png"), dpi=150); plt.show()

# Fig B: <A_v> vs hz
x, y = series(runs, "born", "bulk", "Av_mean")
_, ye = series(runs, "born", "bulk", "Av_sem")
fig, ax = plt.subplots(figsize=(6, 4))
if len(x):
    ax.errorbar(x, y, yerr=ye, fmt="-o", capsize=3)
ax.axhline(1.0, ls=":", c="k", alpha=0.5)
ax.set_xlabel(r"$h_z$")
ax.set_ylabel(r"$\langle A_v\rangle$ (Born ratio mean)")
ax.set_title(f"Vertex-stabilizer expectation vs $h_z$  (L={L})")
ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"08_Av_vs_hz_L{L}.png"), dpi=150); plt.show()

# Fig C: delta distributions across hz (Born, bulk)
fig, ax = plt.subplots(figsize=(7, 4.5))
cmap = plt.cm.viridis
n = max(len(runs) - 1, 1)
for i, r in enumerate(runs):
    blk = r.get("born_bulk")
    if not blk:
        continue
    edges = np.asarray(blk["hist"]["edges"]); counts = np.asarray(blk["hist"]["counts"], float)
    if counts.sum() == 0:
        continue
    c = 0.5 * (edges[:-1] + edges[1:]); w = np.diff(edges)
    ax.plot(c, counts / counts.sum() / w, color=cmap(i / n), label=f"$h_z$={r['hz']:.2f}")
ax.set_xlabel(r"$\delta_v=\log\psi(A_v\sigma)-\log\psi(\sigma)$")
ax.set_ylabel("density (Born, bulk)")
ax.set_yscale("log"); ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
ax.set_xlim(-1, 1)
ax.set_title(f"$\\delta_v$ distribution vs $h_z$  (L={L})")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"08_delta_dist_L{L}.png"), dpi=150); plt.show()